In [37]:
import pandas as pd 
import sqlite3

In [38]:
import httpx

In [101]:
base_uri = 'http://127.0.0.1:8000/api/'

api_routes = {
    1: 'program_university_kpi_latest',
    2: 'program_category',
    3: 'program_noc_link',
    4: 'provincial_noc_group_labor_statistic',
    5: 'noc_grouping',
    6: 'economic_region_employment_estimate',
    7: 'economicregion',
    8: 'province',
    9: 'university',
    10: 'prediction_cache',
    11: 'seasonal_decomposition'
}

# Instantiating a synchronous client with 120s timeout
client = httpx.Client(timeout=500.0)

# Generic function for making a request to the API given a formatted URL
def Make_Request(url: str):
    try:
        print(f"URL => {url}")
        response = client.get(url)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Request failed for {url}: {e}")
        return []

def RequestProvince() -> pd.DataFrame:
    try:
        url_to_call = f"{base_uri}{api_routes[8]}/"
        data = Make_Request(url=url_to_call)
        return pd.DataFrame(data)
    except Exception as e:
        print(f"RequestProvince() => {e}")
        return pd.DataFrame()

def RequestNOCGroupings() -> pd.DataFrame:
    try:
        url_to_call = f"{base_uri}{api_routes[5]}/"
        data = Make_Request(url=url_to_call)
        return pd.DataFrame(data)
    except Exception as e:
        print(f"RequestProvince() => {e}")
        return pd.DataFrame()

def RequestionSeasonalDecomp(provinceid):
    try:
        url_to_call = f"{base_uri}{api_routes[11]}/?provinceid={provinceid}"
        data = Make_Request(url=url_to_call)
        return pd.DataFrame(data)

    except Exception as e:
        print(f"RequestSeasonalDecompo() => {e}")
        return pd.DataFrame()

#region Request Functions

In [102]:
df = pd.read_parquet('Data/SeasonalDecomposedDataClustered.parquet')

In [103]:
df_NOC_Grouping = RequestNOCGroupings()
df_NOC_Grouping

URL => http://127.0.0.1:8000/api/noc_grouping/


,noc_groupingid,noc_grouping,unnamed_2
0,1,Professional occupations in health [31],None
1,2,"Sales and service occupations, except manageme...",None
2,3,Supervisors and occupations in natural resourc...,None
3,4,Helpers and labourers and other transport driv...,None
4,5,"Workers and labourers in natural resources, ag...",None
5,6,Nursing and allied health professionals [313],None
6,7,Retail sales and service supervisors and speci...,None
7,8,Professional occupations in finance [111],None
8,9,Administrative occupations and transportation ...,None
9,10,Health treating and consultation services prof...,None


In [104]:
NOCGroupingID_NOC_Group_Map = df_NOC_Grouping.set_index('noc_groupingid')['noc_grouping'].to_dict()

In [105]:
df_Provinces = RequestProvince()
df_Provinces

URL => http://127.0.0.1:8000/api/province/


,provinceid,province_dguid,province,province_shorthand
0,59,2021A000259,British Columbia,BC
1,24,2021A000224,Quebec,QC
2,11,2021A000211,Prince Edward Island,PE
3,47,2021A000247,Saskatchewan,SK
4,46,2021A000246,Manitoba,MB
5,13,2021A000213,New Brunswick,NB
6,10,2021A000210,Newfoundland and Labrador,NL
7,60,2021A000260,Yukon,YT
8,61,2021A000261,Northwest Territories,NT
9,48,2021A000248,Alberta,AB


In [106]:
ProvinceID_Province_Map = df_Provinces.set_index('provinceid')['province'].to_dict()

In [107]:
df['noc_grouping'] = df['noc_groupingid'].map(NOCGroupingID_NOC_Group_Map)
df['province'] = df['provinceid'].map(ProvinceID_Province_Map)

In [108]:
df

,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds,provinceid,noc_groupingid,Trend Cluster,noc_grouping,province
0,7.0,7.378831,-0.433604,-0.109472,-0.171635,0.335879,2000-01-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador
1,8.2,7.410238,0.542999,-0.063244,0.075139,0.234868,2000-02-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador
2,7.1,7.440902,0.183164,0.135241,-0.264254,-0.395053,2000-03-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador
3,7.6,7.470413,0.108099,-0.047459,0.179786,-0.110839,2000-04-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador
4,6.9,7.497976,-0.678920,-0.077763,-0.156244,0.314951,2000-05-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador
...,...,...,...,...,...,...,...,...,...,...,...,...
181381,15.9,16.568508,-0.454525,-0.237140,-0.991029,1.014186,2025-05-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia
181382,19.8,17.148172,1.821578,0.557542,-0.774380,1.047088,2025-06-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia
181383,17.5,17.745231,0.064598,-0.269077,-0.662542,0.621789,2025-07-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia
181384,18.4,18.357159,-0.750631,-0.561815,1.249591,0.105696,2025-08-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia


In [109]:
df['entryid'] = df.index.values + 1
df

,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds,provinceid,noc_groupingid,Trend Cluster,noc_grouping,province,entryid
0,7.0,7.378831,-0.433604,-0.109472,-0.171635,0.335879,2000-01-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador,1
1,8.2,7.410238,0.542999,-0.063244,0.075139,0.234868,2000-02-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador,2
2,7.1,7.440902,0.183164,0.135241,-0.264254,-0.395053,2000-03-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador,3
3,7.6,7.470413,0.108099,-0.047459,0.179786,-0.110839,2000-04-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador,4
4,6.9,7.497976,-0.678920,-0.077763,-0.156244,0.314951,2000-05-01,10.0,1,11,Professional occupations in health [31],Newfoundland and Labrador,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
181381,15.9,16.568508,-0.454525,-0.237140,-0.991029,1.014186,2025-05-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia,181382
181382,19.8,17.148172,1.821578,0.557542,-0.774380,1.047088,2025-06-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia,181383
181383,17.5,17.745231,0.064598,-0.269077,-0.662542,0.621789,2025-07-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia,181384
181384,18.4,18.357159,-0.750631,-0.561815,1.249591,0.105696,2025-08-01,59.0,60,9,Occupations in front-line public protection se...,British Columbia,181385


In [110]:
conn = sqlite3.connect('../EducationPlanningDB.db')

In [111]:
df.to_sql('Tbl_Seasonal_Decomposition', conn, if_exists="replace", index=False)

181386

Testing 

In [100]:
SeasonlDecompSample = RequestionSeasonalDecomp(provinceid=35)
SeasonlDecompSample

URL => http://127.0.0.1:8000/api/seasonal_decomposition/?provinceid=35


,entryid,datestamp,provinceid,province,trend_cluster,noc_groupingid,noc_grouping,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12
0,3091,2000-01-01 00:00:00,35,Ontario,2,1,Professional occupations in health [31],153.1,153.584406,-1.489052,0.849931,-2.853015,3.007730
1,3092,2000-01-01 00:00:00,35,Ontario,6,1,Professional occupations in health [31],153.1,153.584406,-1.489052,0.849931,-2.853015,3.007730
2,3093,2000-02-01 00:00:00,35,Ontario,2,1,Professional occupations in health [31],151.7,152.835585,-0.580377,0.655482,-2.430219,1.219528
3,3094,2000-02-01 00:00:00,35,Ontario,6,1,Professional occupations in health [31],151.7,152.835585,-0.580377,0.655482,-2.430219,1.219528
4,3095,2000-03-01 00:00:00,35,Ontario,2,1,Professional occupations in health [31],145.9,152.090875,0.689873,-1.341210,-2.165538,-3.374000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36457,360296,2025-07-01 00:00:00,35,Ontario,1,60,Occupations in front-line public protection se...,45.8,48.697155,1.733771,0.471393,-0.180243,-4.922076
36458,360297,2025-08-01 00:00:00,35,Ontario,5,60,Occupations in front-line public protection se...,39.8,48.182645,-3.680010,-0.090930,-0.384035,-4.227670
36459,360298,2025-08-01 00:00:00,35,Ontario,1,60,Occupations in front-line public protection se...,39.8,48.182645,-3.680010,-0.090930,-0.384035,-4.227670
36460,360299,2025-09-01 00:00:00,35,Ontario,5,60,Occupations in front-line public protection se...,44.8,47.658644,0.233695,-0.572767,3.019928,-5.539501
